# Package Counter - 接件及时率 (Ketepatan Waktu Penerimaan Paket)

Notebook ini punya **2 bagian**:

1. **Processing Harian** - proses 1 file Excel (misalnya laporan 1 hari)
2. **Processing Bulk** - proses banyak file Excel sekaligus dari 1 folder, hasilnya digabung jadi 1 file output

Kedua bagian pakai fungsi inti yang sama (`process_one_file`), supaya logikanya konsisten.

> Lihat `README.md` di folder yang sama untuk penjelasan lengkap input, output, dan cara pakai.

In [ ]:
# Jalankan sekali saja kalau package belum ke-install
import sys

!{sys.executable} -m pip install pandas openpyxl

## 1. Konfigurasi

**Ganti path di bawah ini sesuai lokasi file kamu.** Ini satu-satunya bagian yang perlu diubah setiap kali dipakai.

In [ ]:
from pathlib import Path

# ============================================================
# KONFIGURASI - EDIT BAGIAN INI
# ============================================================

# --- Untuk PROCESSING HARIAN (1 file) ---
DAILY_INPUT_FILE = Path(r"C:\Users\devin\Documents\PackageCounter\接件及时率\14+接件及时率明细-实时64b84f36-0c1c-4c44-943d-fdf47f77f48f.xlsx")
DAILY_OUTPUT_FILE = Path(r"C:\Users\devin\Documents\PackageCounter\接件及时率\output\daily_processed.xlsx")

# --- Untuk PROCESSING BULK (banyak file dalam 1 folder) ---
BULK_INPUT_FOLDER = Path(r"C:\Users\devin\Documents\PackageCounter\接件及时率\bulk_input")
BULK_FILE_PATTERN = "*.xlsx"          # pola nama file yang mau diproses
BULK_OUTPUT_FILE = Path(r"C:\Users\devin\Documents\PackageCounter\接件及时率\output\bulk_processed_master.xlsx")

# ============================================================

# Pastikan folder output ada
DAILY_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
BULK_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Daily input :", DAILY_INPUT_FILE, "| exists:", DAILY_INPUT_FILE.exists())
print("Bulk folder :", BULK_INPUT_FOLDER, "| exists:", BULK_INPUT_FOLDER.exists())

## 2. Fungsi Inti

`process_one_file()` membaca 1 file Excel mentah dan menghasilkan rekap per (`查询日期`, `代理区名称`, `网点名称`). Dipakai oleh bagian Harian maupun Bulk — logikanya sama persis dengan versi asli, cuma dirapikan.

In [ ]:
import pandas as pd
import numpy as np
import gc
import time

REQUIRED_COLUMNS = [
    "查询日期",
    "代理区名称",
    "网点名称",
    "运单号",
    "网点卸车扫描时间",
    "责任主体",
    "是否及时（件）",
]

GROUP_COLS = [
    "查询日期",
    "代理区名称",
    "网点名称",
]

COUNT_COLUMNS = [
    "实接件数",
    "未接件数",
    "接件及时件数",
    "不及时件数",
    "jumlah paket tepat waktu",
    "jumlah keseluruhan paket exclude GW, include outlet dan NULL",
]

FINAL_COLUMN_ORDER = [
    "查询日期",
    "代理区名称",
    "网点名称",
    "应接件数",
    "实接件数",
    "留仓率",
    "未接件数",
    "接件及时件数",
    "不及时件数",
    "接件及时率（件）",
    "网点接件及时率（件）",
    "jumlah paket tepat waktu",
    "jumlah keseluruhan paket exclude GW, include outlet dan NULL",
]


def process_one_file(file_path):
    """Proses 1 file Excel mentah -> rekap per (查询日期, 代理区名称, 网点名称)."""

    start_time = time.time()
    file_path = Path(file_path)

    print("=" * 60)
    print(f"Processing: {file_path.name}")
    print("=" * 60)

    # Baca hanya kolom yang diperlukan. header=1 karena baris pertama
    # di file sumber adalah baris deskripsi filter, bukan header kolom.
    df = pd.read_excel(
        file_path,
        sheet_name=0,
        header=1,
        usecols=REQUIRED_COLUMNS,
        engine="openpyxl",
    )

    print(f"Raw rows: {len(df):,}")

    # Ubah string kosong jadi NaN
    for col in ["代理区名称", "网点名称", "责任主体", "是否及时（件）"]:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

    # 1. 应接件数 - jumlah 运单号
    result = (
        df.groupby(GROUP_COLS, dropna=False)
        .agg(应接件数=("运单号", "count"))
        .reset_index()
    )

    # 2. 实接件数 - 网点卸车扫描时间 tidak kosong
    actual_mask = df["网点卸车扫描时间"].notna()
    actual = (
        df.assign(_count=actual_mask.astype("int8"))
        .groupby(GROUP_COLS, dropna=False)["_count"]
        .sum()
        .reset_index(name="实接件数")
    )
    result = result.merge(actual, on=GROUP_COLS, how="left")

    # 3. 未接件数
    result["未接件数"] = result["应接件数"] - result["实接件数"]

    # 4. 留仓率
    result["留仓率"] = np.where(
        result["应接件数"] != 0,
        result["未接件数"] / result["应接件数"],
        np.nan,
    )

    # 5. 接件及时件数 (责任主体=网点 & 是否及时（件）=是)
    timely_mask = df["责任主体"].eq("网点") & df["是否及时（件）"].eq("是")
    timely = (
        df.assign(_count=timely_mask.astype("int8"))
        .groupby(GROUP_COLS, dropna=False)["_count"]
        .sum()
        .reset_index(name="接件及时件数")
    )
    result = result.merge(timely, on=GROUP_COLS, how="left")

    # 6. 不及时件数 (责任主体=网点 & 是否及时（件）=否)
    late_mask = df["责任主体"].eq("网点") & df["是否及时（件）"].eq("否")
    late = (
        df.assign(_count=late_mask.astype("int8"))
        .groupby(GROUP_COLS, dropna=False)["_count"]
        .sum()
        .reset_index(name="不及时件数")
    )
    result = result.merge(late, on=GROUP_COLS, how="left")

    # 7. 接件及时率（件）
    result["接件及时率（件）"] = np.where(
        result["应接件数"] != 0,
        result["接件及时件数"] / result["应接件数"],
        np.nan,
    )

    # 8. Total paket exclude 分拨 (NULL tetap dihitung)
    not_division_mask = df["责任主体"].ne("分拨")
    total_non_gw = (
        df.assign(_count=not_division_mask.astype("int8"))
        .groupby(GROUP_COLS, dropna=False)["_count"]
        .sum()
        .reset_index(name="jumlah keseluruhan paket exclude GW, include outlet dan NULL")
    )
    result = result.merge(total_non_gw, on=GROUP_COLS, how="left")

    # 9. jumlah paket tepat waktu (责任主体!=分拨 & 是否及时（件）=是)
    non_gw_timely_mask = df["责任主体"].ne("分拨") & df["是否及时（件）"].eq("是")
    non_gw_timely = (
        df.assign(_count=non_gw_timely_mask.astype("int8"))
        .groupby(GROUP_COLS, dropna=False)["_count"]
        .sum()
        .reset_index(name="jumlah paket tepat waktu")
    )
    result = result.merge(non_gw_timely, on=GROUP_COLS, how="left")

    # 10. 网点接件及时率（件）
    total_col = "jumlah keseluruhan paket exclude GW, include outlet dan NULL"
    result["网点接件及时率（件）"] = np.where(
        result[total_col] != 0,
        result["jumlah paket tepat waktu"] / result[total_col],
        np.nan,
    )

    # Isi kolom hitung yang kosong dengan 0
    for col in COUNT_COLUMNS:
        result[col] = result[col].fillna(0).astype("int64")

    # Urutan kolom final
    result = result[FINAL_COLUMN_ORDER]

    elapsed = time.time() - start_time
    print(f"Result rows: {len(result):,}")
    print(f"Processing time: {elapsed:.1f} detik")

    del df
    gc.collect()

    return result

## 3. Bagian 1 — Processing Harian (1 file)

Proses 1 file yang sudah di-set di `DAILY_INPUT_FILE`, lalu simpan ke `DAILY_OUTPUT_FILE`.

In [ ]:
daily_result = process_one_file(DAILY_INPUT_FILE)

print("Total baris outlet:", len(daily_result))
display(daily_result)

In [ ]:
daily_result.to_excel(DAILY_OUTPUT_FILE, index=False)

print(f"Output tersimpan: {DAILY_OUTPUT_FILE}")
print(f"Total baris: {len(daily_result):,}")

## 4. Bagian 2 — Processing Bulk (banyak file sekaligus)

Proses semua file yang cocok `BULK_FILE_PATTERN` di dalam `BULK_INPUT_FOLDER`, gabungkan hasilnya jadi satu tabel, lalu simpan ke `BULK_OUTPUT_FILE`.

Kalau ada 1 file gagal diproses (misalnya format beda / rusak), file itu di-skip dengan pesan warning, dan file lainnya tetap lanjut diproses.

In [ ]:
def process_bulk_folder(folder_path, pattern="*.xlsx"):
    """Proses semua file di folder_path yang cocok pattern, gabungkan jadi 1 DataFrame."""

    folder_path = Path(folder_path)
    files = sorted(folder_path.glob(pattern))

    if not files:
        raise FileNotFoundError(
            f"Tidak ada file yang cocok '{pattern}' di dalam {folder_path}"
        )

    print(f"Ditemukan {len(files)} file untuk diproses.\n")

    all_results = []
    failed_files = []

    for f in files:
        try:
            all_results.append(process_one_file(f))
        except Exception as e:
            print(f"[GAGAL] {f.name}: {e}")
            failed_files.append(f.name)
        print()

    if not all_results:
        raise RuntimeError("Semua file gagal diproses, tidak ada hasil.")

    combined = pd.concat(all_results, ignore_index=True)

    print("=" * 60)
    print(f"Selesai. File berhasil: {len(all_results)}/{len(files)}")
    if failed_files:
        print(f"File gagal: {failed_files}")
    print(f"Total baris gabungan: {len(combined):,}")
    print("=" * 60)

    return combined

In [ ]:
bulk_result = process_bulk_folder(BULK_INPUT_FOLDER, BULK_FILE_PATTERN)

display(bulk_result)

In [ ]:
bulk_result.to_excel(BULK_OUTPUT_FILE, index=False)

print(f"Output tersimpan: {BULK_OUTPUT_FILE}")
print(f"Total baris: {len(bulk_result):,}")